# NTT Deep Dive: I/O Contract, Operation Taxonomy, and Hardware Execution

> **ECE-9413 Assignment 1 — Companion Analysis Notebook**  
> Covers three things the tutorial notebook doesn't spell out explicitly:
> 1. The **exact I/O contract** `ntt` must satisfy (dtypes, shapes, values — with worked examples)
> 2. A **taxonomy of every operation** in the algorithm (vec⊙vec, vec⊙scalar, gather, sequential stage loop)
> 3. **How CPUs execute it today** vs **how GPUs/TPUs can execute it** — and why the gap is large

---

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from sympy import isprime

jax.config.update("jax_enable_x64", True)

print("JAX version :", jax.__version__)
print("Backend     :", jax.default_backend())
print("Devices     :", jax.devices())

---
## Section 1 — The NTT I/O Contract

### What the harness calls

```python
# Called ONCE before timing — cost is not measured:
psi_powers, twiddles = prepare_tables(
    q          = q,           # uint32 scalar
    psi_powers = psi_powers,  # uint32[N]
    twiddles   = twiddles,    # uint32[N]
)

# Called in the hot loop — this IS what gets timed:
y = ntt(
    x,                        # uint32[batch, N] — input coefficients
    q          = q,           # uint32 scalar
    psi_powers = psi_powers,  # uint32[N] — twist table
    twiddles   = twiddles,    # uint32[N] — Stockham twiddle table
)
```

Every parameter has a **precise contract**. Wrong dtype, wrong shape, or mixing up `psi_powers` with `twiddles` produces wrong outputs — and the test suite will catch it.

---
### 1.1 — Input: `x` (polynomial coefficients)

| Property | Value |
|---|---|
| Type | `jnp.ndarray` |
| dtype | `jnp.uint32` |
| Shape | `(batch, N)` — 2-D array; `N` must be a power of two |
| Value range | Every element is in `[0, q-1]` — already reduced mod `q` |
| Interpretation | `x[b, n]` = the `n`-th coefficient of the `b`-th polynomial |

**Why batch?**  
In practice, cryptographic protocols need to NTT many polynomials under the same ring parameters (same `q`, `N`, `ψ`). Batching lets the GPU process all of them in parallel with a single kernel launch, amortising launch overhead over `batch` polynomials.

**Why uint32 and not int32?**  
All values are in `[0, q)` — always non-negative. Using unsigned avoids signed-integer overflow when two values are added before reduction (e.g. `a + b` where `a = q-1`, `b = q-1` gives `2q-2 ≤ 2×2^31 < 2^32`).

In [ ]:
# ── Concrete example: valid x for a (batch=3, N=8) NTT ───────────────────────
N       = 8
BATCH   = 3
Q_small = 97   # tiny prime where (q-1) % 2N = 96 % 16 = 0

x_demo = jnp.array([
    [0, 1, 2, 3,  4,  5,  6,  7],   # batch row 0
    [7, 6, 5, 4,  3,  2,  1,  0],   # batch row 1
    [0, 1, 0, 1, 96, 95, 48, 48],   # batch row 2
], dtype=jnp.uint32)

print("x contract check")
print("=" * 50)
print(f"  dtype : {x_demo.dtype}   (must be uint32)")
print(f"  shape : {x_demo.shape}  (must be (batch, N) = ({BATCH}, {N}))")
print(f"  min   : {int(x_demo.min())}")
print(f"  max   : {int(x_demo.max())}   (must be < q = {Q_small})")
assert x_demo.dtype == jnp.uint32
assert x_demo.shape == (BATCH, N)
assert int(x_demo.max()) < Q_small
print("  All checks PASS")

print("\nEach row is one polynomial's coefficients:")
for b in range(BATCH):
    poly_str = " + ".join(f"{int(x_demo[b,n])}·x^{n}" for n in range(N) if x_demo[b,n] > 0)
    if not poly_str:
        poly_str = "0"
    print(f"  x[{b}]: {x_demo[b].tolist()}  →  p_{b}(x) = {poly_str[:60]}...")

---
### 1.2 — Input: `q` (prime modulus)

| Property | Value |
|---|---|
| Type | `int` or `jnp.uint32` scalar |
| Value | A prime satisfying `(q - 1) % (2N) == 0` |
| Bit-length | < 31 bits (`q < 2^31 ≈ 2.1 billion`) |
| Role | All arithmetic is mod `q`; also determines which primitive root `ψ` exists |

**Why `(q - 1) % 2N == 0`?**  
We need a primitive `2N`-th root of unity `ψ` modulo `q`. By Fermat's little theorem, the multiplicative group `(Z/qZ)*` has order `q - 1`. A primitive `2N`-th root exists **if and only if** `2N` divides `q - 1`.  
This is why `provided.generate_ntt_modulus(N)` searches for primes of the form `q = k × 2N + 1`.

**Why < 31 bits?**  
Multiplying two values in `[0, q)` gives a product up to `(q-1)^2 < (2^31)^2 = 2^62`, which fits in a `uint64` intermediate. This lets us implement `mod_mul` with a single 64-bit multiply + one modular reduction.

In [ ]:
# The moduli the benchmark generates (via generate_ntt_modulus)
def find_ntt_prime(N, bit_length=31):
    """Find largest prime < 2^bit_length where (q-1) % 2N == 0."""
    step = 2 * N
    limit = (1 << bit_length) - 1
    k = (limit - 1) // step
    candidate = k * step + 1
    while candidate >= 2:
        if isprime(candidate):
            return candidate
        candidate -= step
    raise RuntimeError("No prime found")

print(f"{'N':>8}  {'log2(N)':>7}  {'q':>14}  {'q-1 = k×2N':>12}  {'bits':>5}")
print("-" * 55)
for logn in [8, 10, 12, 14, 16, 20]:
    N_test = 1 << logn
    q_test = find_ntt_prime(N_test)
    k      = (q_test - 1) // (2 * N_test)
    bits   = q_test.bit_length()
    print(f"  {N_test:6d}  {logn:7d}  {q_test:14d}  k={k:>6}        {bits:5d}")

print()
# Show why 31-bit limit lets us use uint64 for intermediates
q_max = find_ntt_prime(1 << 20)
product_max = (q_max - 1) ** 2
print(f"Largest benchmark q       = {q_max}")
print(f"(q-1)^2                   = {product_max}")
print(f"Fits in uint32?           {product_max < 2**32}   ← OVERFLOW — must use uint64")
print(f"Fits in uint64?           {product_max < 2**64}   ← safe")

---
### 1.3 — Input: `psi_powers` (negacyclic twist table)

| Property | Value |
|---|---|
| Type | `jnp.ndarray` |
| dtype | `jnp.uint32` |
| Shape | `(N,)` — 1-D array of length exactly `N` |
| Value | `psi_powers[n] = ψ^n mod q` |
| Source | `provided.precompute_tables(N, q, psi)` |

**What is `ψ` (psi)?**  
`ψ` is a **primitive 2N-th root of unity** mod `q`:  
- `ψ^{2N} ≡ 1 (mod q)` — cycles back after 2N steps  
- `ψ^N ≡ -1 (mod q)` — exactly half-way is negation ← **the negacyclic property**

**How `psi_powers` is used:**  
The first step in the NTT is the **negacyclic twist**:  
```
x_twisted[b, n] = x[b, n] · psi_powers[n]  (mod q)
```
This converts a negacyclic NTT into a standard cyclic NTT on `x_twisted`.

**Important:** After `prepare_tables`, the array may be in a different representation (e.g., Montgomery form). The values won't literally be `ψ^n mod q` anymore, but they'll be mathematically equivalent in the transformed domain.

In [ ]:
# Build psi_powers from scratch for a small example
from sympy import factorint

def find_generator(q):
    phi = q - 1
    factors = list(factorint(phi).keys())
    for g in range(2, q):
        if all(pow(g, phi // f, q) != 1 for f in factors):
            return g
    raise ValueError

def find_primitive_root_of_unity(order, q):
    assert (q - 1) % order == 0
    g = find_generator(q)
    return pow(g, (q - 1) // order, q)

N_ex  = 8
q_ex  = 97    # (97-1) % 16 = 0 ✓
psi_ex = find_primitive_root_of_unity(2 * N_ex, q_ex)

# Build psi_powers: psi_powers[n] = psi^n mod q
psi_powers_ex = np.array([pow(psi_ex, n, q_ex) for n in range(N_ex)], dtype=np.uint32)

print(f"N={N_ex}, q={q_ex}, ψ={psi_ex}")
print(f"Verification: ψ^N = ψ^{N_ex} = {pow(psi_ex, N_ex, q_ex)}  (should be {q_ex-1} = -1 mod {q_ex})")
print()
print("psi_powers table:")
print(f"  shape = {psi_powers_ex.shape}  dtype = {psi_powers_ex.dtype}")
print(f"  values = {psi_powers_ex.tolist()}")
print()
print("Key entries:")
print(f"  psi_powers[0] = ψ^0 = {psi_powers_ex[0]}  (always 1)")
print(f"  psi_powers[1] = ψ^1 = {psi_powers_ex[1]}  (= ψ itself)")
print(f"  psi_powers[N//2] = ψ^{N_ex//2} = {psi_powers_ex[N_ex//2]}  (the halfway point)")
print(f"  psi_powers[N-1] = ψ^{N_ex-1} = {psi_powers_ex[N_ex-1]}")
print()
# Show the twist effect on a simple input
x_ex = np.array([1, 1, 1, 1, 1, 1, 1, 1], dtype=np.uint32)  # all ones
x_twisted = (x_ex.astype(np.int64) * psi_powers_ex.astype(np.int64)) % q_ex
print("Twist effect on x = [1,1,1,1,1,1,1,1]:")
print(f"  x         = {x_ex.tolist()}")
print(f"  psi_powers= {psi_powers_ex.tolist()}")
print(f"  x_twisted = {x_twisted.tolist()}   (= x[n] * psi^n mod q)")

---
### 1.4 — Input: `twiddles` (Stockham twiddle table)

| Property | Value |
|---|---|
| Type | `jnp.ndarray` |
| dtype | `jnp.uint32` |
| Shape | `(N,)` — same length as `psi_powers` |
| Layout | `twiddles[span + j] = ω^{j × stride}  mod q` where `ω = ψ²`, `span = 2^s`, `stride = N / (2 × span)` |
| Source | `provided.precompute_tables(N, q, psi)` |

**What are twiddle factors?**  
In the Stockham butterfly at stage `s`, each pair `(u, v)` is combined as:
```
top = u + W × v   (mod q)
bot = u - W × v   (mod q)
```
where `W = twiddles[span + j]` for position `j` within the current span. The twiddle factor rotates `v` by the appropriate power of `ω` before combining.

**Index layout — critical detail:**  
```
twiddles[0]       → unused (index 0)
twiddles[1]       → stage s=0, span=1, j=0  (one twiddle: ω^{N/2})
twiddles[2..3]    → stage s=1, span=2, j=0,1
twiddles[4..7]    → stage s=2, span=4, j=0..3
...                 ...
twiddles[N/2..N-1]→ stage s=log2(N)-1, span=N/2, j=0..N/2-1
```
This layout means stage `s` reads exactly `twiddles[span : 2*span]` — a contiguous slice.

In [ ]:
# Build twiddles from scratch and show the full layout
def build_twiddles(N, q, psi):
    """Build Stockham twiddle table. twiddles[span+j] = omega^(j*stride) mod q."""
    omega  = pow(psi, 2, q)   # omega = psi^2 is the N-th root of unity
    stages = N.bit_length() - 1  # log2(N)
    twiddles = np.ones(N, dtype=np.uint32)
    for s in range(stages):
        span   = 1 << s
        stride = N // (2 * span)   # stride in the omega exponent
        step   = pow(omega, stride, q)  # omega^stride
        cur    = 1
        for j in range(span):
            twiddles[span + j] = cur
            cur = (cur * step) % q
    return twiddles

twiddles_ex = build_twiddles(N_ex, q_ex, psi_ex)
omega_ex    = pow(psi_ex, 2, q_ex)
stages_ex   = N_ex.bit_length() - 1

print(f"N={N_ex}, q={q_ex}, ψ={psi_ex}, ω=ψ²={omega_ex}")
print(f"stages = log2({N_ex}) = {stages_ex}")
print()
print("Twiddle table layout:")
print(f"  twiddles = {twiddles_ex.tolist()}")
print()

for s in range(stages_ex):
    span   = 1 << s
    stride = N_ex // (2 * span)
    step   = pow(omega_ex, stride, q_ex)
    slice_ = twiddles_ex[span : 2 * span].tolist()
    print(f"  Stage s={s}: span={span:2d}, stride={stride:2d}, ω^stride={step:3d}")
    print(f"    twiddles[{span}:{2*span}] = {slice_}")
    # Verify manually
    expected = [pow(omega_ex, stride * j, q_ex) for j in range(span)]
    assert slice_ == expected, f"Mismatch at stage {s}"

print()
print("All stage twiddle slices verified.")
print()
print("Key: stage s reads exactly twiddles[2^s : 2^(s+1)] — always a contiguous slice.")
print("     This is what makes the Stockham layout cache-friendly on both CPU and GPU.")

---
### 1.5 — Input: `prepare_tables` (optional one-time hook)

| Property | Value |
|---|---|
| Called | **Once**, before timing starts |
| Signature | `prepare_tables(*, q, psi_powers, twiddles) → (psi_powers, twiddles)` |
| Cost | Not timed — can be arbitrarily expensive |
| Purpose | Convert tables to a faster representation (e.g., Montgomery form) |

**Contract:** The returned `(psi_powers, twiddles)` must be accepted by your `ntt` without any further conversion. The harness passes them directly to every subsequent `ntt` call.

**Common uses:**
- Convert to **Montgomery form** — eliminates `%` (division) from the hot path
- Pre-broadcast twiddle slices into a `(stages, max_span)` layout for faster indexing
- Pad or re-layout for a specific reshape pattern

---
### 1.6 — Output: `y` (NTT coefficients)

| Property | Value |
|---|---|
| Type | `jnp.ndarray` |
| dtype | `jnp.uint32` |
| Shape | `(batch, N)` — same shape as input `x` |
| Value range | Every element is in `[0, q-1]` — reduced mod `q` |
| Interpretation | `y[b, k]` = the `k`-th NTT coefficient of the `b`-th polynomial |

**Mathematical definition:**
```
y[b, k] = Σ_{n=0}^{N-1}  x[b, n] · ψ^{(2k+1)·n}   (mod q)
```

**Invariant the tests check:**
- `y.shape == x.shape` — same shape
- `y.dtype == jnp.uint32` — exact dtype
- `y.min() >= 0` and `y.max() < q` — all outputs in range
- `y == reference_ntt(x)` — exact match against SymPy oracle

**Linearity property (useful self-test):**  
`NTT(a + b mod q) ≡ NTT(a) + NTT(b) mod q` — all operations are linear over `Z/qZ`.

In [ ]:
# ── Worked end-to-end example matching the tutorial intro ─────────────────────
# We implement a reference NTT in pure Python to show every step

def mod_add(a, b, q):
    return ((a.astype(jnp.uint64) + b.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)

def mod_sub(a, b, q):
    q64 = jnp.uint64(q)
    return ((a.astype(jnp.uint64) + q64 - b.astype(jnp.uint64)) % q64).astype(jnp.uint32)

def mod_mul(a, b, q):
    return ((a.astype(jnp.uint64) * b.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)

def ntt_ref(x, *, q, psi_powers, twiddles):
    """Reference NTT: Stockham algorithm with negacyclic twist."""
    batch, N = x.shape
    stages   = N.bit_length() - 1
    # Step 1: negacyclic twist
    out = mod_mul(x, psi_powers[None, :], q)
    # Step 2: Stockham stages
    for s in range(stages):
        span       = 1 << s
        num_blocks = N // (2 * span)
        W          = twiddles[span : 2 * span]        # shape (span,)
        out_4d     = out.reshape(batch, num_blocks, 2, span)
        u, v       = out_4d[:, :, 0, :], out_4d[:, :, 1, :]
        W_bc       = W[None, None, :]
        Wv         = mod_mul(v, W_bc, q)
        top        = mod_add(u, Wv, q)
        bot        = mod_sub(u, Wv, q)
        out        = jnp.stack([top, bot], axis=2).reshape(batch, N)
    return out

# Direct negacyclic formula (O(N^2) naive, for verification)
def ntt_naive(x_row, *, q, psi):
    N = len(x_row)
    return [
        sum(int(x_row[n]) * pow(psi, (2*k+1)*n, q) for n in range(N)) % q
        for k in range(N)
    ]

# Run on N=8, q=97
pp_jnp = jnp.asarray(psi_powers_ex, dtype=jnp.uint32)
tw_jnp = jnp.asarray(twiddles_ex,   dtype=jnp.uint32)
Q_jnp  = jnp.uint32(q_ex)

x_test = jnp.array([[1, 2, 3, 4, 5, 6, 7, 8]], dtype=jnp.uint32)

y_ref  = ntt_naive(x_test[0].tolist(), q=q_ex, psi=psi_ex)
y_mine = ntt_ref(x_test, q=Q_jnp, psi_powers=pp_jnp, twiddles=tw_jnp)

print("OUTPUT SHAPES & DTYPES")
print("=" * 50)
print(f"  Input  x   : shape={x_test.shape}  dtype={x_test.dtype}")
print(f"  Output y   : shape={y_mine.shape}  dtype={y_mine.dtype}")
print(f"  Values: {y_mine[0].tolist()}")
print(f"  Range : [{int(y_mine.min())}, {int(y_mine.max())}]  (must be [0, {q_ex-1}])")
print()
print("CORRECTNESS vs NAIVE O(N²):")
print(f"  Naive  : {y_ref}")
print(f"  Stockham: {y_mine[0].tolist()}")
print(f"  Match  : {y_mine[0].tolist() == y_ref}")

---
### 1.7 — I/O Summary Table

```
INPUTS
────────────────────────────────────────────────────────────────────────
x            uint32[batch, N]   Polynomial coefficients; all values in [0, q)
q            uint32 scalar      Prime modulus; (q-1) % (2N) == 0; q < 2^31
psi_powers   uint32[N]          psi_powers[n] = ψ^n mod q  (before prepare_tables)
twiddles     uint32[N]          twiddles[span+j] = ω^(j×stride) mod q

OPTIONAL HOOK (not timed)
────────────────────────────────────────────────────────────────────────
prepare_tables(q, psi_powers, twiddles) → (psi_powers', twiddles')
                                          Converts tables to faster form (e.g. Montgomery)

OUTPUTS
────────────────────────────────────────────────────────────────────────
y            uint32[batch, N]   NTT output; y[b,k] = Σ x[b,n]·ψ^{(2k+1)n} mod q

WHERE:
  N      = transform size (power of 2)
  ψ      = primitive 2N-th root of unity mod q  (ψ^N ≡ -1 mod q)
  ω = ψ² = primitive N-th root of unity mod q   (ω^N ≡  1 mod q)
  stages = log2(N)   (number of Stockham stages)
```

---
## Section 2 — Operation Taxonomy: Every Op Mapped to Hardware Primitives

Understanding *what kind* of operation each step is determines whether it benefits from GPU/TPU acceleration.  
We use 5 categories:

| Category | Notation | Description | Example |
|---|---|---|---|
| **Vec ⊙ Vec** | `V[N] op V[N] → V[N]` | Elementwise op on two same-shape arrays | `u + Wv` pointwise |
| **Vec ⊙ Scalar** | `V[N] op s → V[N]` | Broadcast a scalar across an array | `v * W` where W is one twiddle |
| **Gather/Reshape** | `V[N] → V[N]` (reinterpreted) | Non-copy view change | `out.reshape(...)`, `arr[span:2*span]` |
| **Stack/Scatter** | `(V[K], V[K]) → V[N]` | Write two arrays into an interleaved result | `jnp.stack([top, bot])` |
| **Sequential loop** | per-stage → per-stage | One stage depends on previous stage output | outer `for s in range(stages)` |

**Crucially: SumCheck has REDUCTIONS (`jnp.sum`) — NTT does NOT.**  
NTT is entirely elementwise and reshape operations. This makes it even more parallelism-friendly than SumCheck.

**Also crucially: NTT tables stay the same size every stage.**  
SumCheck tables halve each round. NTT always works on the full `(batch, N)` array — so GPU parallelism is equally large at every stage.

---
### 2.1 — Operation Map: Every Step in `ntt`

```
STEP 0 — Negacyclic Twist
────────────────────────────────────────────────────────────────────────
  out = mod_mul(x, psi_powers[None, :], q)
    psi_powers[None, :] shape: (1, N) → broadcasts to (batch, N)
    Operation type: VEC ⊙ VEC   (each x[b,n] independently multiplied by psi_powers[n])
    Array size: (batch, N)   — full array, fully parallel
    Intermediate: (batch, N) uint64 products, then mod, then back to uint32

  Note: this is the ONLY place psi_powers is used.
  Equivalently: x_twisted[b,n] = x[b,n] · ψ^n mod q

STEP 1 — Per-Stage: Extract Twiddle Slice (log2(N) stages)
────────────────────────────────────────────────────────────────────────
  W = twiddles[span : 2*span]    ← GATHER/SLICE  V[N] → V[span]
    This is a zero-copy slice (view) of the twiddle array.
    Size: span elements, where span = 2^s, grows from 1 to N/2.

STEP 2 — Per-Stage: Reshape to 4-D (log2(N) stages)
────────────────────────────────────────────────────────────────────────
  out_4d = out.reshape(batch, num_blocks, 2, span)    ← RESHAPE  (metadata only, FREE)
    num_blocks = N / (2*span) = N / (2·2^s)
    axis 0: batch
    axis 1: which block (num_blocks blocks of size 2*span)
    axis 2: upper (0) vs lower (1) half of each block
    axis 3: position within the half (0..span-1)
  u = out_4d[:, :, 0, :]    ← VIEW of upper halves, shape (batch, num_blocks, span)
  v = out_4d[:, :, 1, :]    ← VIEW of lower halves, shape (batch, num_blocks, span)

  This reshape is the Stockham trick: u and v for all blocks are in contiguous memory.

STEP 3 — Per-Stage: Butterfly (log2(N) stages)
────────────────────────────────────────────────────────────────────────
  W_bc = W[None, None, :]          ← RESHAPE  (1, 1, span) — broadcast shape
  Wv   = mod_mul(v, W_bc, q)       ← VEC ⊙ VEC  (span broadcast over num_blocks)
  top  = mod_add(u, Wv, q)         ← VEC ⊙ VEC  (batch × num_blocks × span ops)
  bot  = mod_sub(u, Wv, q)         ← VEC ⊙ VEC

  Total parallel ops per stage: batch × N/2 butterflies, each is 1 mul + 1 add + 1 sub
  Array size: (batch, num_blocks, span) = (batch, N/2) elements per u and v
  → Always operates on FULL array, NOT halving like SumCheck!

STEP 4 — Per-Stage: Stockham Write-Back (log2(N) stages)
────────────────────────────────────────────────────────────────────────
  out = jnp.stack([top, bot], axis=2).reshape(batch, N)  ← STACK + RESHAPE
    jnp.stack creates (batch, num_blocks, 2, span)
    reshape collapses to (batch, N)
    This is the Stockham out-of-place write: top halves and bot halves
    go to contiguous locations in the new buffer, keeping access coalesced.

STEP 5 — Outer Stage Loop
────────────────────────────────────────────────────────────────────────
  for s in range(stages):    ← SEQUENTIAL (log2(N) iterations)
    Each stage reads out[] from the previous stage → cannot parallelise.
    But: log2(N) ≤ 20 for any practical N — the loop is tiny.
```

In [ ]:
# ── Demonstrate each operation category in isolation ─────────────────────────
Q  = jnp.uint32(97)
N_demo = 8
BATCH_demo = 2

key = jax.random.PRNGKey(42)
x_d = jax.random.randint(key, (BATCH_demo, N_demo), 0, 97, dtype=jnp.uint32)
pp_d = jnp.asarray(psi_powers_ex, dtype=jnp.uint32)
tw_d = jnp.asarray(twiddles_ex,   dtype=jnp.uint32)

print("Operation taxonomy demonstration (N=8, batch=2, q=97)")
print("=" * 65)

# STEP 0: VEC ⊙ VEC twist
psi_row = pp_d[None, :]          # (1, N) broadcast ready
out = mod_mul(x_d, psi_row, Q)  # (batch, N) × (1, N) → (batch, N)
print(f"STEP 0  VEC⊙VEC  Twist: mod_mul(x, psi_powers)")
print(f"  x[0]        = {x_d[0].tolist()}")
print(f"  psi_powers  = {pp_d.tolist()}")
print(f"  x_twisted[0]= {out[0].tolist()}")
print(f"  All {BATCH_demo}×{N_demo} = {BATCH_demo*N_demo} elements computed independently")
print()

# STEP 1: slice twiddles (free view)
s = 1; span = 1 << s
W_slice = tw_d[span : 2*span]
print(f"STEP 1  GATHER   Twiddle slice for stage s={s}: twiddles[{span}:{2*span}]")
print(f"  W = {W_slice.tolist()}  (zero-copy view, no data moved)")
print()

# STEP 2: reshape (free metadata change)
num_blocks = N_demo // (2 * span)
out_4d = out.reshape(BATCH_demo, num_blocks, 2, span)
u = out_4d[:, :, 0, :]
v = out_4d[:, :, 1, :]
print(f"STEP 2  RESHAPE  out ({BATCH_demo},{N_demo}) → 4D ({BATCH_demo},{num_blocks},2,{span})")
print(f"  u[0] (upper halves of all blocks) = {u[0].tolist()}")
print(f"  v[0] (lower halves of all blocks) = {v[0].tolist()}")
print(f"  Reshape is metadata-only: O(1) cost regardless of N")
print()

# STEP 3: butterfly (VEC ⊙ VEC, 3 operations)
W_bc = W_slice[None, None, :]
Wv   = mod_mul(v, W_bc, Q)
top  = mod_add(u, Wv, Q)
bot  = mod_sub(u, Wv, Q)
print(f"STEP 3  VEC⊙VEC  Butterfly: 1×mod_mul + 1×mod_add + 1×mod_sub")
print(f"  Wv[0]  = {Wv[0].tolist()}  (W*v)")
print(f"  top[0] = {top[0].tolist()}  (u + Wv)")
print(f"  bot[0] = {bot[0].tolist()}  (u - Wv)")
print(f"  {BATCH_demo}×{num_blocks}×{span} = {BATCH_demo*num_blocks*span} butterflies, all independent")
print()

# STEP 4: stack + reshape
new_out = jnp.stack([top, bot], axis=2).reshape(BATCH_demo, N_demo)
print(f"STEP 4  STACK+RESHAPE  Write-back to (batch, N)")
print(f"  Output[0] = {new_out[0].tolist()}")
print(f"  Contiguous memory layout → coalesced GPU writes")

---
### 2.2 — Operation Count as a Function of N and batch

Let $N$ = transform size, $B$ = batch, $S$ = `log2(N)` stages.

| Step | Operation type | Count (elementwise ops across full array) |
|---|---|---|
| Twist | VEC ⊙ VEC mul | $B \times N$ |
| Per stage: twiddle slice | Gather (free view) | 0 |
| Per stage: reshape to 4D | Reshape (free) | 0 |
| Per stage: `mod_mul(v, W)` | VEC ⊙ VEC mul | $B \times N/2$ |
| Per stage: `mod_add(u, Wv)` | VEC ⊙ VEC add | $B \times N/2$ |
| Per stage: `mod_sub(u, Wv)` | VEC ⊙ VEC sub | $B \times N/2$ |
| Per stage: stack + reshape | Stack + Reshape | $B \times N$ (write) |

**Total elementwise multiply ops:**  
$B \times N$ (twist) $+$ $S \times B \times N/2$ (butterfly muls) $= B \times N \times (1 + S/2)$

**Contrast with SumCheck:** SumCheck tables *halve* each round → geometric series → 2× work in first round. NTT tables *never change size* → the same $B \times N/2$ butterfly work occurs at **every** stage.

**This is why NTT scales better on GPU:** every stage is equally large, so GPU occupancy stays high from stage 0 to stage $S-1$.

In [ ]:
def count_ntt_ops(logn, batch):
    """Count elementwise operations per stage and total for NTT."""
    N      = 1 << logn
    stages = logn
    rows   = []

    # Twist (stage 0 equivalent)
    twist_muls = batch * N

    # Per-stage butterfly counts
    for s in range(stages):
        muls = batch * (N // 2)  # mod_mul(v, W)
        adds = batch * (N // 2)  # mod_add(u, Wv)
        subs = batch * (N // 2)  # mod_sub(u, Wv)
        rows.append((s, muls, adds, subs, muls + adds + subs))

    return twist_muls, rows

logn_ex = 6
batch_ex = 4
twist_ops, stage_rows = count_ntt_ops(logn_ex, batch_ex)
N_ex2 = 1 << logn_ex

print(f"Operation counts: N=2^{logn_ex}={N_ex2}, batch={batch_ex}")
print()
print(f"Twist:  {twist_ops:,} mod_mul ops (one per coefficient × batch)")
print()
print(f"{'Stage':6s}  {'Muls':>8s}  {'Adds':>8s}  {'Subs':>8s}  {'Total ops':>12s}  {'% of grand total'}")
print("-" * 70)

grand_total = twist_ops + sum(r[4] for r in stage_rows)
print(f"  twist  {twist_ops:>8,}                              {twist_ops:>12,}  {100*twist_ops/grand_total:5.1f}%")
for s, muls, adds, subs, total in stage_rows:
    pct = 100 * total / grand_total
    print(f"  s={s}    {muls:>8,}  {adds:>8,}  {subs:>8,}  {total:>12,}  {pct:5.1f}%")
print("-" * 70)
print(f"  TOTAL  {'':>8s}  {'':>8s}  {'':>8s}  {grand_total:>12,}  100.0%")
print()
print("Key insight: EVERY stage processes the same number of ops.")
print("GPU occupancy is equally high from stage 0 to stage log2(N)-1.")
print("(Compare to SumCheck: Round 1 does 50% of all work; later rounds shrink.)")

---
### 2.3 — Why There Are No Reductions, and What That Means

The SumCheck protocol requires `jnp.sum()` calls (REDUCTION operations) to compute `claim0` and each `g(t)`. Reductions collapse `N` elements to 1 scalar — they require a global synchronisation across all threads.

**NTT has zero reductions.** Every output element `y[b, k]` depends on all input elements `x[b, n]`, but this fan-in is implemented through the butterfly network — not through a sum.

**Why the butterfly avoids the reduction bottleneck:**
- Reduction: `N` elements → 1 scalar, O(log N) time with parallel hardware
- Butterfly stage: `N` elements → `N` elements, O(1) time with N/2 parallel units

Each butterfly operates on a pair independently. No element needs to wait for results from a different pair within the same stage. The only dependency is **between stages** — and there are just `log2(N)` of them.

```
SumCheck:   Σ over 2^n elements   →  1 scalar    (reduction, O(log N) serial steps)
NTT stage:  N/2 independent pairs →  N elements  (no reduction, O(1) with N/2 hardware)
```

**Implication:** NTT is more parallelism-friendly than SumCheck for the same reason that sorting networks beat comparison-based sorting for GPU — the work is structured as independent pairs, not a global accumulation.

In [ ]:
# Demonstrate: trace every data dependency in one NTT stage
# For N=8, stage s=0 (span=1): pairs are (0,1), (2,3), (4,5), (6,7)

N_trace = 8
print(f"Data dependencies in NTT stage s=0 (span=1, N={N_trace}):")
print()

span = 1
num_blocks = N_trace // (2 * span)
pairs = [(blk * 2 * span + j, blk * 2 * span + span + j)
         for blk in range(num_blocks)
         for j in range(span)]

print(f"  {len(pairs)} butterfly pairs — ALL INDEPENDENT (can execute in parallel):")
for i, (u_idx, v_idx) in enumerate(pairs):
    print(f"  Pair {i}: out[{u_idx}], out[{v_idx}]  →  new_out[{u_idx}], new_out[{v_idx}]")

print()
print("Compare: a reduction sum over 8 elements requires 3 serial steps:")
print("  Step 1: (a0+a1), (a2+a3), (a4+a5), (a6+a7)  — 4 parallel adds")
print("  Step 2: ((a0+a1)+(a2+a3)), ((a4+a5)+(a6+a7)) — 2 parallel adds")
print("  Step 3: final sum                              — 1 add")
print("  Total: 3 serial synchronisation points (= log2(8))")
print()
print("NTT stage: 0 serial synchronisation points (all pairs independent).")
print("Only synchronisation is BETWEEN stages (log2(N) barriers total).")

---
## Section 3 — CPU vs GPU/TPU: How the Hardware Executes Each Operation

This section explains *concretely* what happens inside the hardware for each operation, and why the performance gap is large.

---
### 3.1 — How a CPU Executes NTT Today

A modern CPU (e.g., Apple M2 or Intel Xeon) runs the NTT as follows:

#### Step-by-step CPU execution of one NTT stage (N=2^20, batch=4):

```
Input: (4, 2^20) uint32 array = 4 × 4 MB = 16 MB total

CPU SIMD (vectorized, 1 core, AVX2):
  Register width: 256-bit → holds 4× uint64 values
  mod_mul(v, W): promote to uint64, multiply, mod, demote → 4 elements/cycle
  mod_add(u, Wv): similar, 4 elements/cycle
  Total butterfly throughput ≈ 4 elements/cycle per core

With 8 cores:
  Effective throughput ≈ 32 elements/cycle
  At 3 GHz: ~96 Gelem/s theoretical
  Real throughput (memory-bound, 16 MB > L3): ~3–5 Gelem/s

For N=2^20, batch=4: 4 × 2^20 / 2 = 2M butterflies per stage
At 3 Gelem/s: ~0.7 ms per stage
With log2(2^20) = 20 stages: ~14 ms total
```

#### CPU execution of the stage loop:

```python
for s in range(stages):          ← Python interpreter (20 iterations; tiny overhead)
    W = twiddles[span:2*span]    ← zero-copy NumPy slice
    out_4d = out.reshape(...)    ← zero-copy metadata change
    u = out_4d[:, :, 0, :]      ← zero-copy view
    v = out_4d[:, :, 1, :]      ← zero-copy view
    Wv = mod_mul(v, W_bc, q)    ← JAX dispatches to XLA CPU kernel (hot!)
    top = mod_add(u, Wv, q)     ← JAX dispatches to XLA CPU kernel (hot!)
    bot = mod_sub(u, Wv, q)     ← JAX dispatches to XLA CPU kernel (hot!)
    out = jnp.stack(...).reshape(...)  ← XLA CPU kernel
```

**Key CPU bottlenecks:**
1. **Memory bandwidth:** For N=2^20, batch=4, each stage reads and writes 16 MB. The L3 cache (32 MB) can hold one full pass, but barely — 3 variables (v, Wv, u) exceed it.
2. **Lack of parallelism across stages:** The 20-stage loop is fully sequential — each stage's output feeds the next.
3. **`%` operator:** The modular reduction in `mod_mul` hides a division operation, which is ~20–40 cycles on modern CPUs.

---
### 3.2 — How a GPU Executes NTT

A GPU (e.g., NVIDIA A100) has a radically different architecture:

```
A100 specs (relevant to NTT):
  Streaming Multiprocessors (SMs): 108
  CUDA cores per SM: 64
  Total CUDA cores: 6,912
  Clock speed: ~1.4 GHz
  HBM2e bandwidth: 2 TB/s
  L2 cache: 40 MB
```

#### GPU execution of one NTT stage (N=2^20, batch=4):

```
With jax.jit, XLA fuses the butterfly (mod_mul + mod_add + mod_sub) into one kernel:

  Kernel launch: 4 × 2^20 / 2 = 2M threads, each handling one butterfly pair
  Thread blocks: 2M / 256 = 8192 blocks
  All 6,912 CUDA cores active simultaneously

  Each thread:
    - Reads u[idx] and v[idx] from HBM (if not in L2)
    - Reads W[j] from L2 (twiddle slice is small, stays cached)
    - Computes Wv = v * W (uint64 multiply + mod)
    - Computes top = u + Wv, bot = u - Wv
    - Writes top and bot to new buffer in HBM

For N=2^20, batch=4: 16 MB fits in GPU L2 (40 MB) → very few HBM misses!
HBM bandwidth used: ~16 MB read + ~16 MB write per stage = 32 MB/stage
At 2 TB/s: ~16 μs per stage (memory-bound)
For 20 stages: ~320 μs ≈ 0.32 ms total
```

**The key differences vs CPU:**
1. **6,912 vs 32 parallel units** — ~200× more butterfly pairs processed simultaneously
2. **L2 cache (40 MB) holds the entire working set for N≤2^20** — almost no HBM traffic
3. **jax.jit fuses** `mod_mul + mod_add + mod_sub` into one kernel — 3× less HBM traffic vs unfused

#### Kernel fusion (jax.jit) matters:

```
Without JIT (3 separate kernels per butterfly):
  Kernel 1: Wv = mod_mul(v, W)    → write Wv to HBM
  Kernel 2: top = mod_add(u, Wv)  → read Wv from HBM, write top to HBM
  Kernel 3: bot = mod_sub(u, Wv)  → read Wv, u from HBM, write bot to HBM
  Total HBM traffic: ~5 reads + ~3 writes per butterfly

With JIT (1 fused kernel):
  Single kernel: read u, v, W → compute all three in registers → write top, bot
  Total HBM traffic: 3 reads + 2 writes per butterfly
  Speedup from fusion: ~40% less memory traffic → ~40% faster when memory-bound
```

---
### 3.3 — How a TPU Executes NTT

A TPU is optimized for matrix multiplications and is **less naturally suited** to NTT:

```
TPU v4 specs:
  Matrix Multiply Units (MXUs): 4
  MXU systolic array: 128×128 bfloat16
  Peak throughput: 275 TFLOPS (matmul only)
  HBM bandwidth: 1.2 TB/s
```

**NTT on TPU:**
- The MXUs are **idle** — NTT has no matrix multiplications. This is worse for TPU than for SumCheck (which also has no matmul, but at least uses simpler element types).
- The NTT butterfly is `mod_mul + mod_add + mod_sub` over `uint32`, which TPU handles on its scalar cores (not MXUs).
- Advantage: XLA on TPU is **very aggressive at operator fusion** — the full butterfly chain may become one op.
- The twiddle layout (contiguous slices per stage) works well on TPU because XLA can tile the computation.

**Can we map NTT to matmul on TPU?**  
Yes, with `O(N × log N / block_size)` matmuls, each of size `block_size × block_size`. This is the approach taken by some GPU NTT papers (e.g., [Fast GPU NTT #2](https://arxiv.org/pdf/2407.13055)) — map a few butterfly stages at a time to a shared-memory matrix multiply. On TPU, XLA may do something similar automatically.

**Bottom line:** For standard JAX NTT, GPU > TPU. If you reformulate NTT as batched matmuls, TPU can be competitive.

---
### 3.4 — Operation-Level CPU vs GPU Comparison

```
OPERATION              CPU (8 cores, AVX2)         GPU (A100)              Speedup
────────────────────────────────────────────────────────────────────────────────────
Twist: mod_mul(x,pp)   ~32 elem/cycle              ~6912 elem/cycle        ~216×
(batch×N muls)         L3 cache if N≤2^20          L2 cache if N≤2^20      ←same L2

Butterfly mod_mul      ~32 elem/cycle              ~6912 elem/cycle        ~216×
(batch×N/2 muls)       memory-bound at large N     L2 hit at N=2^20        better

Butterfly mod_add/sub  ~64 elem/cycle (faster)     ~6912 elem/cycle        ~108×
                       add is faster than mul       same cores, faster      

Twiddle slice[s:2s]    zero-copy slice              zero-copy slice         1×
                       cache-line aligned           cache-line aligned      same

reshape/stack          zero-copy (metadata)         zero-copy (metadata)    1×
                       no data moved                no data moved           same

stage loop (log2N)     Python sequential            Python sequential       1×
(20 iter for N=2^20)   ~20 × Python overhead       ~20 × kernel launch     similar
                                                    (JIT eliminates these!)

L2 working set fit     N=2^20 × 4 batch = 16 MB    N=2^20 × 4 = 16 MB     
                       L3=32MB (tight, 3 arrays)    L2=40MB (all fits!)     GPU wins
```

**The pattern:**
- For butterfly ops (which are ~100% of compute): GPU is ~100–200× faster
- For reshape/slice (zero-copy): same on both
- For the stage loop (log2(N) Python iters): `jax.jit` makes this irrelevant by tracing the whole function
- For cache: GPU L2 (40 MB) fits the whole `(batch=4, N=2^20)` array; CPU L3 (32 MB) barely doesn't

**NTT vs SumCheck on GPU:**  
NTT has a constant per-stage array size → GPU occupancy is high at every stage.  
SumCheck tables halve each round → GPU occupancy drops to nearly 0 in final rounds.  
**NTT is a better fit for GPU than SumCheck.**

In [ ]:
import time

N_bench  = 2**16
BATCH_bench = 4
q_bench  = find_ntt_prime(N_bench)
Q_bench  = jnp.uint32(q_bench)

key  = jax.random.PRNGKey(0)
a_b  = jax.random.randint(key, (BATCH_bench, N_bench), 0, q_bench, dtype=jnp.uint32)
b_b  = jax.random.randint(key, (BATCH_bench, N_bench), 0, q_bench, dtype=jnp.uint32)
w_b  = jax.random.randint(key, (N_bench // 2,),       1, q_bench, dtype=jnp.uint32)

jit_mul = jax.jit(lambda a, b: mod_mul(a, b, Q_bench))
jit_add = jax.jit(lambda a, b: mod_add(a, b, Q_bench))

# Fused butterfly: W*v, then u±Wv
u_b = a_b[:, :N_bench//2]
v_b = a_b[:, N_bench//2:]
W_bc = w_b[None, :]
jit_butterfly = jax.jit(lambda u, v, W: (
    mod_add(u, mod_mul(v, W, Q_bench), Q_bench),
    mod_sub(u, mod_mul(v, W, Q_bench), Q_bench)
))

# Warm up
_ = jit_mul(a_b, b_b).block_until_ready()
_ = jit_add(a_b, b_b).block_until_ready()
_ = jit_butterfly(u_b, v_b, W_bc)[0].block_until_ready()

RUNS = 20

def bench(fn, *args, label):
    t0 = time.perf_counter()
    for _ in range(RUNS):
        fn(*args).block_until_ready() if not isinstance(fn(*args), tuple) else [x.block_until_ready() for x in fn(*args)]
    ms = (time.perf_counter() - t0) / RUNS * 1000
    gelem = (BATCH_bench * N_bench) / (ms / 1000) / 1e9
    print(f"  {label:35s}  {ms:7.3f} ms   {gelem:.2f} Gelem/s")

def bench_tuple(fn, *args, label, n_elem):
    t0 = time.perf_counter()
    for _ in range(RUNS):
        r = fn(*args)
        r[0].block_until_ready()
        r[1].block_until_ready()
    ms = (time.perf_counter() - t0) / RUNS * 1000
    gelem = n_elem / (ms / 1000) / 1e9
    print(f"  {label:35s}  {ms:7.3f} ms   {gelem:.2f} Gelem/s")

print(f"Benchmark: batch={BATCH_bench}, N=2^16={N_bench:,}, backend={jax.default_backend()}")
print("=" * 75)
bench(jit_mul, a_b, b_b, label="mod_mul (twist, VEC⊙VEC)")
bench(jit_add, a_b, b_b, label="mod_add (butterfly add, VEC⊙VEC)")
bench_tuple(jit_butterfly, u_b, v_b, W_bc, label="Full butterfly (fused, JIT)", n_elem=BATCH_bench*N_bench)

---
### 3.5 — Memory Hierarchy: Where the Data Lives

```
CPU MEMORY HIERARCHY (approximate)
──────────────────────────────────────────────────────
L1 cache   32 KB per core    4 cycles    2 TB/s per core
L2 cache   256 KB per core   12 cycles   1 TB/s per core
L3 cache   32 MB shared      40 cycles   200 GB/s
DRAM       unlimited         200 cycles  50 GB/s

(batch=4, N=2^20) array = 4 × 4 MB = 16 MB
→ Fits in L3 (32 MB), but only barely with two copies (old + new buffer)
  3 arrays (u, Wv, out) = 24 MB → exceeds L3 → DRAM traffic

(batch=4, N=2^24) array = 4 × 64 MB = 256 MB
→ Way beyond L3 → all accesses go to DRAM (50 GB/s)

GPU MEMORY HIERARCHY (A100)
──────────────────────────────────────────────────────
Registers    per thread   0 cycles      per-thread
Shared mem   164 KB/SM    ~32 cycles    19 TB/s
L2 cache     40 MB        ~200 cycles   12 TB/s
HBM2e DRAM   80 GB        ~500 cycles   2 TB/s

(batch=4, N=2^20) = 16 MB  → fits in GPU L2 (40 MB) → all stage accesses hit L2!
(batch=4, N=2^23) = 128 MB → exceeds L2 → goes to HBM (still 2 TB/s)
(batch=4, N=2^24) = 256 MB → goes to HBM (2 TB/s vs CPU's 50 GB/s → 40× faster!)
```

**Critical insight:** For N ≤ 2^20, the GPU L2 cache (40 MB) holds the **entire working set across all log2(N) stages**. The CPU L3 (32 MB) does not — so the GPU gets an extra advantage beyond raw parallelism: **dramatically higher effective bandwidth**.

**Why `jax.jit` helps so much for NTT:**  
Without JIT, each `mod_mul`, `mod_add`, `mod_sub` is a separate kernel call. Each kernel must read from HBM and write back to HBM. XLA fusion merges the three butterfly operations into one kernel, keeping the intermediate `Wv` in registers — never touching HBM at all.

In [ ]:
import math

print("NTT working-set size and cache tier at each N")
print(f"batch=4, dtype=uint32 (4 bytes)")
print()
print(f"{'logN':>5}  {'N':>8}  {'Size (MB)':>10}  {'CPU L3 (32MB)':>14}  {'GPU L2 (40MB)':>14}  {'GPU HBM bw win':>15}")
print("-" * 75)

BATCH_m = 4
for logn in [12, 14, 16, 18, 20, 22, 24]:
    N_m    = 1 << logn
    size_MB = BATCH_m * N_m * 4 / 1e6
    cpu_tier = "L3" if size_MB < 32 else "DRAM"
    gpu_tier = "L2"  if size_MB < 40 else "HBM"
    # HBM (2 TB/s) vs DRAM (50 GB/s) → 40× bandwidth advantage when both in DRAM
    bw_win = "40×" if cpu_tier == "DRAM" and gpu_tier == "HBM" else ("L2 hit!" if gpu_tier == "L2" else "—")
    print(f"  {logn:3d}  {N_m:8,}  {size_MB:10.1f}  {cpu_tier:>14}  {gpu_tier:>14}  {bw_win:>15}")

print()
print("For N=2^20 (benchmark default): GPU L2 cache holds everything → near-zero HBM traffic.")
print("CPU L3 is tight at 16 MB — 3 arrays (u, Wv, out) = 24 MB → some DRAM spill.")

---
### 3.6 — The Stage Loop: Why It's Not a Bottleneck (Unlike SumCheck's Round Loop)

SumCheck has an outer loop over `num_rounds = n` — the number of Boolean variables. For large `n = 20`, that's 20 Python-level iterations, each with a kernel launch, Python overhead, and a `jnp.sum` reduction.

NTT has an outer loop over `stages = log2(N)` — but with `jax.jit` on the whole function, **the Python loop is traced away entirely**. XLA sees the full computation as a static graph and compiles all stages into a single optimised program.

```
WITHOUT jax.jit (eager mode):
  Python stage loop → log2(N) kernel launches
  Each launch: ~10–100 μs Python + CUDA overhead
  For N=2^20: 20 × 50 μs = 1 ms of avoidable overhead

WITH jax.jit (traced mode):
  Python loop runs ONCE during tracing (at JIT compile time)
  XLA generates one compiled program for all stages
  At runtime: no Python overhead, no per-stage kernel launch
  All stages fused into a minimal sequence of GPU kernels
```

**Why SumCheck is harder to JIT the stage loop:**  
In SumCheck, the prover receives `challenges` as dynamic inputs — they change every call. The table update depends on the challenge value. XLA can still trace this, but the `for round_idx` loop with dynamic-length `challenges` array is trickier. In NTT, the twiddle factors are **static** (precomputed, passed as captured constants in the JIT) — the XLA compiler can specialise for them completely.

In [ ]:
import time

# ── Compare eager vs JIT for the full NTT ────────────────────────────────────
N_jit  = 2**14
B_jit  = 4
q_jit  = find_ntt_prime(N_jit)
psi_jit = find_primitive_root_of_unity(2 * N_jit, q_jit)

pp_jit_np, tw_jit_np = [np.array([pow(psi_jit, n, q_jit) for n in range(N_jit)], dtype=np.uint32),
                         None]

# build twiddles
omega_jit = pow(psi_jit, 2, q_jit)
stages_jit = N_jit.bit_length() - 1
tw_np = np.ones(N_jit, dtype=np.uint32)
for s in range(stages_jit):
    span = 1 << s
    stride = N_jit // (2 * span)
    step = pow(omega_jit, stride, q_jit)
    cur = 1
    for j in range(span):
        tw_np[span + j] = cur
        cur = (cur * step) % q_jit

pp_jit = jnp.asarray(pp_jit_np, dtype=jnp.uint32)
tw_jit = jnp.asarray(tw_np,     dtype=jnp.uint32)
Q_jit  = jnp.uint32(q_jit)

key  = jax.random.PRNGKey(99)
x_jit = jax.random.randint(key, (B_jit, N_jit), 0, q_jit, dtype=jnp.uint32)

# Eager: no JIT
_ = ntt_ref(x_jit, q=Q_jit, psi_powers=pp_jit, twiddles=tw_jit).block_until_ready()  # warm-up
RUNS = 10
t0 = time.perf_counter()
for _ in range(RUNS):
    ntt_ref(x_jit, q=Q_jit, psi_powers=pp_jit, twiddles=tw_jit).block_until_ready()
t_eager = (time.perf_counter() - t0) / RUNS * 1000

# JIT
ntt_jit = jax.jit(lambda z: ntt_ref(z, q=Q_jit, psi_powers=pp_jit, twiddles=tw_jit))
t_compile_start = time.perf_counter()
_ = ntt_jit(x_jit).block_until_ready()
t_compile = (time.perf_counter() - t_compile_start) * 1000

t0 = time.perf_counter()
for _ in range(RUNS):
    ntt_jit(x_jit).block_until_ready()
t_jit = (time.perf_counter() - t0) / RUNS * 1000

throughput = B_jit * N_jit / (t_jit / 1000) / 1e6

print(f"N=2^14={N_jit:,}, batch={B_jit}, backend={jax.default_backend()}")
print(f"stages = log2({N_jit}) = {stages_jit}")
print()
print(f"Eager (no JIT):      {t_eager:.2f} ms/call")
print(f"JIT compile time:    {t_compile:.0f} ms (one-time cost)")
print(f"JIT steady-state:    {t_jit:.3f} ms/call")
print(f"JIT speedup:         {t_eager/t_jit:.1f}×")
print(f"Throughput (JIT):    {throughput:.2f} Mcoeff/s")
print()
print(f"JIT eliminates {stages_jit} Python-level kernel launches per call.")
print(f"XLA traces the static stage loop and compiles all {stages_jit} stages into one program.")

---
### 3.7 — Amdahl's Law: The Stage Loop Is Not Amdahl-Limited (With JIT)

For **SumCheck**, the `num_rounds` loop is an **Amdahl bottleneck**: even if all table ops run infinitely fast, you still pay `n` Python-level iterations. For `n=20`, this is ~1–2 ms of irreducible overhead.

For **NTT**, the `stages` loop with `jax.jit` is **NOT an Amdahl bottleneck** — it is traced once and compiled away. After JIT compilation:

```
NTT stages = log2(N) = 20 for N=2^20
Without JIT: 20 Python iterations, 20 kernel launches → ~1 ms overhead
With JIT:    0 Python iterations at runtime           → ~0 overhead

The sequential fraction after JIT ≈ 0
Amdahl speedup limit ≈ 1/0 → ∞ (practically: bounded by memory bandwidth)
```

The only true bottleneck is **hardware**: either memory bandwidth (large N) or kernel launch latency (very small N where compute time < launch overhead).

**Contrast with SumCheck:** SumCheck's outer loop cannot be JIT-traced in the same way because it has dynamic challenge values (not known at trace time). Even with JIT, the `num_rounds` outer loop may require Python-level iteration — creating an irreducible Amdahl bottleneck.

This is why NTT can be ~100–200× faster on GPU than on CPU, while SumCheck may only achieve ~50–100× (the sequential fraction limits it further).

In [ ]:
import math

print("NTT: Amdahl's Law estimate — impact of JIT on sequential fraction")
print()
print("Model:")
print("  GPU butterfly throughput: 100× faster than CPU")
print("  Per-stage kernel launch overhead: 50 μs (without JIT)")
print("  With JIT: 0 sequential overhead (stage loop compiled away)")
print()
print(f"{'logN':>5}  {'N':>8}  {'stages':>6}  {'CPU (ms)':>10}  {'GPU eager (ms)':>14}  {'GPU JIT (ms)':>14}  {'Speedup eager':>14}  {'Speedup JIT':>12}")
print("-" * 95)

BATCH_amd = 4
for logn in [8, 12, 16, 20, 24]:
    N_amd     = 1 << logn
    stages    = logn
    # CPU: butterfly at ~1 Gelem/s, batch*N/2 ops per stage, log2(N) stages
    # Total butterfly ops: stages * batch * N/2 muls + same for add/sub = stages * batch * N * 1.5
    cpu_table_ms = stages * BATCH_amd * N_amd * 1.5 / 1e9 * 1000
    # GPU: 100× faster butterfly
    gpu_table_ms = cpu_table_ms / 100
    # Sequential overhead (kernel launches, only without JIT)
    seq_ms_eager = stages * 0.05  # 50 μs per stage
    seq_ms_jit   = 0.01           # ~10 μs total for JIT overhead (negligible)
    
    cpu_total_ms        = cpu_table_ms + seq_ms_eager
    gpu_eager_total_ms  = gpu_table_ms + seq_ms_eager
    gpu_jit_total_ms    = gpu_table_ms + seq_ms_jit
    
    speedup_eager = cpu_total_ms / gpu_eager_total_ms
    speedup_jit   = cpu_total_ms / gpu_jit_total_ms
    
    print(f"  {logn:3d}  {N_amd:8,}  {stages:6d}  {cpu_total_ms:10.2f}  {gpu_eager_total_ms:14.2f}  {gpu_jit_total_ms:14.2f}  {speedup_eager:14.1f}×  {speedup_jit:12.1f}×")

print()
print("JIT removes the Amdahl sequential fraction entirely → speedup approaches 100× even at small N.")
print("Eager mode has a sequential floor set by kernel launch overhead (stages × 50 μs).")

---
### 3.8 — Summary: What the GPU/TPU Buys You

```
┌──────────────────────────────────────────────────────────────────────────────────┐
│           OPERATION             CPU         GPU (JIT)   Why GPU Wins            │
├──────────────────────────────────────────────────────────────────────────────────┤
│ Twist: mod_mul(x, psi_powers)   ~20 ms      ~0.2 ms     6912 vs 32 parallel     │
│ (batch=4, N=2^20)               L3 tight    L2 hits!    cache advantage          │
│                                                                                  │
│ Butterfly mod_mul (per stage)   ~10 ms      ~0.1 ms     same                    │
│ Butterfly mod_add/sub           ~5 ms       ~0.05 ms    same                    │
│ All 20 stages combined          ~300 ms     ~3 ms       ~100× real speedup      │
│                                                                                  │
│ Twiddle slice twiddles[s:2s]    free (view) free (view) no difference           │
│ reshape / jnp.stack             free        free        no difference           │
│                                                                                  │
│ Stage loop (log2(N) iters)      ~1 ms eager ~1 ms eager  1× (same)             │
│                                 ~0 JIT      ~0 JIT      JIT removes both!       │
│                                                                                  │
│ Overall N=2^20, batch=4         ~300 ms     ~3 ms       ~100× with JIT          │
└──────────────────────────────────────────────────────────────────────────────────┘

The three things that matter most for GPU NTT performance:
  1. jax.jit → fuses all butterfly ops into one XLA program; erases stage-loop overhead
  2. Stockham layout → coalesced memory accesses; no bank conflicts; L2-friendly
  3. Montgomery form (in prepare_tables) → replaces % with cheap shifts; 2–4× mul speedup

The one thing that limits GPU NTT performance:
  Memory bandwidth — once data doesn't fit in GPU L2 (N > 2^20 for batch=4),
  you're bound by HBM bandwidth (2 TB/s for A100 — still 40× better than CPU DRAM).

NTT vs SumCheck on GPU:
  NTT: constant array size per stage  → high GPU occupancy at EVERY stage
  NTT: no reductions                  → no global sync barriers
  NTT: static twiddles (JIT-fused)    → zero sequential Amdahl fraction
  → NTT is a BETTER GPU workload than SumCheck
```

In [ ]:
# ── Full end-to-end benchmark across multiple N and batch sizes ────────────────
import time

WARMUP = 5
RUNS   = 20
logns  = [8, 10, 12, 14, 16]
batchs = [1, 4]

print(f"Backend: {jax.default_backend()}")
print(f"{'logN':>5}  {'N':>8}  {'batch':>6}  {'compile ms':>12}  {'median ms':>11}  {'p90 ms':>9}  {'Mcoeff/s':>10}")
print("-" * 70)

for logn in logns:
    N_f   = 1 << logn
    q_f   = find_ntt_prime(N_f)
    psi_f = find_primitive_root_of_unity(2 * N_f, q_f)

    # Build tables
    pp_f_np = np.array([pow(psi_f, n, q_f) for n in range(N_f)], dtype=np.uint32)
    omega_f = pow(psi_f, 2, q_f)
    tw_f_np = np.ones(N_f, dtype=np.uint32)
    stages_f = N_f.bit_length() - 1
    for s in range(stages_f):
        sp = 1 << s; stride = N_f // (2 * sp); step = pow(omega_f, stride, q_f); cur = 1
        for j in range(sp):
            tw_f_np[sp + j] = cur; cur = (cur * step) % q_f

    pp_f = jnp.asarray(pp_f_np, dtype=jnp.uint32)
    tw_f = jnp.asarray(tw_f_np, dtype=jnp.uint32)
    Q_f  = jnp.uint32(q_f)

    for B_f in batchs:
        key_f = jax.random.PRNGKey(logn * 100 + B_f)
        x_f   = jax.random.randint(key_f, (B_f, N_f), 0, q_f, dtype=jnp.uint32)

        fn_f  = jax.jit(lambda z: ntt_ref(z, q=Q_f, psi_powers=pp_f, twiddles=tw_f))

        # JIT compile
        t0 = time.perf_counter()
        _  = fn_f(x_f).block_until_ready()
        compile_ms = (time.perf_counter() - t0) * 1000

        for _ in range(WARMUP - 1):
            fn_f(x_f).block_until_ready()

        times_ms = []
        for _ in range(RUNS):
            t0 = time.perf_counter()
            fn_f(x_f).block_until_ready()
            times_ms.append((time.perf_counter() - t0) * 1000)

        med_ms = float(np.median(times_ms))
        p90_ms = float(np.quantile(times_ms, 0.90))
        mcoeff = B_f * N_f / (med_ms / 1000) / 1e6

        print(f"  {logn:3d}  {N_f:8,}  {B_f:6d}  {compile_ms:12.1f}  {med_ms:11.3f}  {p90_ms:9.3f}  {mcoeff:10.2f}")

---
## Final Reference Card

### I/O at a glance
```python
# INPUTS
x          : uint32[batch, N]    # x[b, n] = n-th coefficient of b-th polynomial; values in [0, q)
q          : uint32 scalar       # prime modulus; (q-1) % (2N) == 0; q < 2^31
psi_powers : uint32[N]           # psi_powers[n] = ψ^n mod q  (before prepare_tables)
twiddles   : uint32[N]           # twiddles[span+j] = ω^(j×stride) mod q

# OPTIONAL HOOK (not timed)
prepare_tables(q, psi_powers, twiddles) → (psi_powers', twiddles')
#   convert to Montgomery form, re-layout, pre-broadcast — anything you want

# OUTPUTS
y          : uint32[batch, N]    # y[b,k] = Σ_{n} x[b,n] · ψ^{(2k+1)n} mod q
```

### Operation taxonomy
```
mod_mul(x, psi_powers, q)          →  VEC ⊙ VEC    (twist; dominant op; fully parallel)
mod_mul(v, W_bcast, q)             →  VEC ⊙ VEC    (butterfly W*v; per stage; fully parallel)
mod_add(u, Wv, q)                  →  VEC ⊙ VEC    (butterfly u+Wv)
mod_sub(u, Wv, q)                  →  VEC ⊙ VEC    (butterfly u-Wv)
twiddles[span : 2*span]            →  GATHER       (free slice; stage-s twiddles)
out.reshape(batch, num_blocks, 2, span) → RESHAPE  (free; Stockham view trick)
jnp.stack([top, bot]).reshape(...)  →  STACK       (write-back; contiguous)
for s in range(log2(N))            →  SEQUENTIAL  (log2(N) stages; JIT traces away)

NO REDUCTIONS anywhere (unlike SumCheck which has jnp.sum for claim0 and each g(t))
NO TABLE SIZE CHANGE between stages (unlike SumCheck where tables halve each round)
```

### Key identities
```
ψ^{2N} ≡  1  (mod q)   ← psi is a primitive 2N-th root of unity
ψ^N    ≡ -1  (mod q)   ← the negacyclic property
ω = ψ²                 ← omega is the N-th root for the cyclic Stockham stages
q ≡ 1 (mod 2N)         ← required for ψ to exist mod q

Butterfly:  top = u + W·v  (mod q)
            bot = u - W·v  (mod q)
            W = twiddles[span + j]  at stage s where span = 2^s

Twist:      x_twisted[b, n] = x[b, n] · psi_powers[n]  (mod q)
```

### GPU performance checklist
1. **`jax.jit`** the entire `ntt()` — fuses all stages into one XLA program; eliminates stage-loop overhead
2. **Montgomery form** in `prepare_tables` — replaces `%` with bit ops in every `mod_mul`; 2–4× speedup
3. **Stockham layout** — already in template; out-of-place writes are coalesced; don't switch to Cooley-Tukey
4. **`(batch, N)` arrays** — never Python-loop over batch; GPU parallelises all rows simultaneously
5. **`block_until_ready()`** in benchmarks — JAX is async; always block before timing
6. **Twiddle slices** `twiddles[span:2*span]` are contiguous — GPU cache-line friendly; keep this layout

### Useful links
- NTT Tutorial: https://eprint.iacr.org/2024/585.pdf
- Montgomery Multiplication: https://cp-algorithms.com/algebra/montgomery_multiplication.html
- Fast GPU NTT #1: https://arxiv.org/pdf/2012.01968
- Fast GPU NTT #2: https://arxiv.org/pdf/2407.13055
- JAX JIT docs: https://jax.readthedocs.io/en/latest/jit-compilation.html